<a href="https://colab.research.google.com/github/Esaiasson/Machine_learning_WS_2025/blob/decision_tree/A1/decision_tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
import time
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    f1_score,
    make_scorer
)

In [132]:
folder = ""
obesity_df_train_minmax_path = folder + "obesity_df_train_minmax_preprocessed.csv"
depression_df_train_minmax_path = folder + "depression_df_train_minmax_preprocessed.csv"
congressional_df_train_path = folder + "congressional_voting_preprocessed.csv"
rev_df_train_minmax_path = folder + "rev_df_lrn_minmax_preprocessed.csv"




In [144]:
obesity_df_train_minmax = pd.read_csv(obesity_df_train_minmax_path)
depression_df_train_minmax = pd.read_csv(depression_df_train_minmax_path)
congressional_df_train = pd.read_csv(congressional_df_train_path)
rev_df_train_minmax = pd.read_csv(rev_df_train_minmax_path)


rev_df_train_minmax = rev_df_train_minmax.drop(rev_df_train_minmax[rev_df_train_minmax["Class_encoded"].isna()].index)
rev_df_train_minmax.head()

array([43, 41,  9,  0, 38, 34, 15, 35,  1,  4, 46, 19, 21, 29, 14, 18,  8,
       11, 13, 26, 39, 17, 49, 31, 20, 23,  5,  2, 44, 12, 32, 45, 48, 33,
        3, 10, 24, 22, 25,  7, 27, 16, 30, 40, 42, 36, 28, 37,  6, 47])

In [ ]:
def train_deci_tree_with_grid(df, target_attribute):

  scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='macro'),
    'recall': make_scorer(recall_score, average='macro'),
    'f1': make_scorer(f1_score, average='macro'),
  }

  param_grid = {
      'criterion': ["entropy", "gini"],
      'min_samples_split': range(2,10,1)
  }

  if df.shape[0] <= 1000:
    param_grid["max_depth"] = range(5,15,1)
  elif df.shape[0] > 1000 and df.shape[0] <= 10000:
    param_grid["max_depth"] = range(10,20,1)
  else:
    param_grid["max_depth"] = range(1,50,1)

  x = df.loc[:, df.columns != target_attribute]
  y_raw = df[target_attribute]
  le = LabelEncoder()
  y = le.fit_transform(y_raw)

  start = time.perf_counter()

  tree = DecisionTreeClassifier(random_state=1)

  grid_search = GridSearchCV(
      estimator=tree,
      param_grid=param_grid,
      cv=5,
      scoring=scoring,
      refit='accuracy',
      verbose=True
  )

  grid_search.fit(x,y)

  results = pd.DataFrame(grid_search.cv_results_)
  elapsed = time.perf_counter() - start
  print("best accuracy", grid_search.best_score_)
  print(grid_search.best_estimator_)
  print("Time(s): ", elapsed)
  return results

In [146]:
results_obesity_minmax = train_deci_tree_with_grid(obesity_df_train_minmax, "obesity_level_grouped")
results_depression_minmax = train_deci_tree_with_grid(depression_df_train_minmax, "depression")
results_congressional = train_deci_tree_with_grid(congressional_df_train, "class")
results_rev_minmax = train_deci_tree_with_grid(rev_df_train_minmax, "Class_encoded")

Fitting 5 folds for each of 160 candidates, totalling 800 fits


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

best accuracy 0.2693333333333333
DecisionTreeClassifier(max_depth=14, min_samples_split=7, random_state=1)
Time(s):  888.7730455679994


In [ ]:
def get_top_results(results_df):
  mean_score_metrics = ["mean_test_f1", "mean_test_accuracy", "mean_test_precision", "mean_test_recall"]
  results_df["combined_rank"] = results_df["rank_test_accuracy"] + results_df["rank_test_precision"] + results_df["rank_test_recall"] + results_df["rank_test_f1"]
  results_df_sorted = results_df.sort_values(by=mean_score_metrics, ascending=False)
  param_cols = [col for col in results_df_sorted.columns if 'param_' in col]
  mean_score_metrics.append("combined_rank")
  relevant_cols = mean_score_metrics + param_cols
  results_df_sorted_relevant = results_df_sorted[relevant_cols]

  return results_df_sorted_relevant

In [149]:
processed_results_obesity_minmax = get_top_results(results_obesity_minmax)
processed_results_depression_minmax = get_top_results(results_depression_minmax)
processed_results_congressional = get_top_results(results_congressional)
processed_results_rev_minmax = get_top_results(results_rev_minmax)

In [129]:
processed_results_obesity_minmax.head()

,mean_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_min_samples_split
51,0.746307,0.784359,0.747751,0.747022,6,entropy,16,5
59,0.745427,0.784361,0.746413,0.746614,8,entropy,17,5
67,0.744468,0.783174,0.745508,0.745562,13,entropy,18,5
75,0.744468,0.783174,0.745508,0.745562,13,entropy,19,5
43,0.742160,0.780203,0.744363,0.741975,27,entropy,15,5


In [130]:
processed_results_depression_minmax.head()

,mean_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_min_samples_split
54,0.825257,0.830777,0.826163,0.824631,30,entropy,7,8
55,0.825257,0.830777,0.826163,0.824631,30,entropy,7,9
48,0.825158,0.830687,0.826078,0.824523,38,entropy,7,2
49,0.825158,0.830687,0.826078,0.824523,38,entropy,7,3
50,0.825158,0.830687,0.826078,0.824523,38,entropy,7,4


In [128]:
processed_results_congressional.head()

,mean_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_min_samples_split
4,0.95753,0.958774,0.958199,0.958378,4,entropy,5,6
12,0.95753,0.958774,0.958199,0.958378,4,entropy,6,6
20,0.95753,0.958774,0.958199,0.958378,4,entropy,7,6
28,0.95753,0.958774,0.958199,0.958378,4,entropy,8,6
36,0.95753,0.958774,0.958199,0.958378,4,entropy,9,6


In [150]:
processed_results_rev_minmax.head()

,mean_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_min_samples_split
157,0.232167,0.269333,0.250731,0.261667,10,gini,14,7
158,0.231309,0.268000,0.250064,0.260667,16,gini,14,8
152,0.230167,0.266667,0.250731,0.260000,16,gini,14,2
154,0.229995,0.266667,0.250731,0.259667,18,gini,14,4
153,0.228928,0.265333,0.250064,0.258333,26,gini,14,3
